# Week 3b — A Customer Support Agent in LangGraph

Everything from this week, assembled into one working application: a support agent for **Husky Tech**, a small online electronics store. The agent will look up orders, check return eligibility, answer policy questions, escalate to a human when it should, and remember the conversation across turns.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. The scenario and the data

A real support agent sits in front of an order database and a policy knowledge base. We mock both with dictionaries so we can focus on the agent. The tool code is the only thing that would change in production; the graph would not.

In [ ]:
ORDERS = {
    "HT-1001": {"item": "Wireless headphones", "status": "delivered", "delivered_on": "2026-09-05", "price": 79.99},
    "HT-1002": {"item": "Mechanical keyboard",  "status": "shipped",   "ordered_on": "2026-09-15", "eta": "2026-09-23", "price": 129.00},
    "HT-1003": {"item": "USB-C dock",           "status": "processing","ordered_on": "2026-09-18", "price": 59.50},
}

RETURN_POLICY_DAYS = 30

FAQ = {
    "shipping": "Standard shipping takes 3-5 business days. Orders over $50 ship free.",
    "return":   "Items can be returned within 30 days of delivery for a full refund.",
    "warranty": "All electronics include a one-year manufacturer warranty.",
    "hours":    "Support is available Monday through Friday, 9am to 6pm ET.",
}

## 2. The tools

Four tools, one job each. Remember from Week 2b: the docstring is not a comment; it is how the model decides which tool to call.

In [ ]:
from langchain.tools import tool

@tool
def look_up_order(order_id: str) -> str:
    """Look up an order by its id (e.g. HT-1001) and return its current details."""
    order = ORDERS.get(order_id.upper())
    if order is None:
        return f"No order found with id {order_id}. Ask the customer to double-check it."
    return str(order)

In [ ]:
from datetime import datetime, date

@tool
def check_return_eligibility(order_id: str) -> str:
    """Check whether an order can still be returned under the 30-day policy. Takes the order id."""
    order = ORDERS.get(order_id.upper())
    if order is None:
        return f"No order found with id {order_id}."
    if order["status"] != "delivered":
        return f"Order {order_id} has not been delivered yet, so the return window has not started."
    delivered = datetime.strptime(order["delivered_on"], "%Y-%m-%d").date()
    days = (date.today() - delivered).days
    if days <= RETURN_POLICY_DAYS:
        return f"Eligible: delivered {days} days ago; returns are accepted within {RETURN_POLICY_DAYS} days of delivery."
    return f"Not eligible: delivered {days} days ago, which is past the {RETURN_POLICY_DAYS}-day window."

In [ ]:
#TODO: write search_faq. It takes the customer's question as a string,
# checks whether any FAQ topic appears in the question (lowercase both sides),
# and returns that topic's answer. If nothing matches, return the list of topics.
# Don't forget the @tool decorator and the docstring.


In [ ]:
@tool
def escalate_to_human(reason: str) -> str:
    """Escalate the conversation to a human support agent. Use when the customer is upset, asks for a person, or the available tools cannot resolve the issue. Provide a one-sentence reason."""
    return f"Escalation ticket ESC-1042 created. Reason: {reason}. A human agent will follow up within one business day."

Test the tools directly before handing them to a model, always.

In [ ]:
print(look_up_order.invoke({"order_id": "HT-1002"}))
print(check_return_eligibility.invoke({"order_id": "HT-1001"}))
print(search_faq.invoke({"question": "What are your support hours?"}))
print(escalate_to_human.invoke({"reason": "Customer requested a human agent."}))

## 3. The system prompt

The tools define what the agent *can* do; the system prompt defines what it *should* do. For a customer-facing agent this is where the guardrails live.

In [ ]:
from langchain.messages import SystemMessage

sys_msg = SystemMessage(content="""You are the customer support assistant for Husky Tech, an online electronics store.

Rules:
- Only answer questions about Husky Tech orders, products, and policies. Politely decline anything else.
- Never invent order details. Always use the tools to look up real data.
- If the customer is upset, asks for a person, or the tools cannot resolve the issue, use escalate_to_human.
- Be concise and polite.""")

## 4. The graph

The same agent loop you wired in W03a.2: assistant node, tool node, `tools_condition`, and the edge back — only the tools are new:

![](figures/graph_support_agent.png)

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

tools = [look_up_order, check_return_eligibility, search_faq, escalate_to_human]
model_with_tools = model.bind_tools(tools)

def assistant(state: MessagesState) -> dict:
    return {"messages": [model_with_tools.invoke([sys_msg] + state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

support_agent = builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(support_agent.get_graph().draw_mermaid_png()))

## 5. Test drives

In [ ]:
from langchain.messages import HumanMessage

result = support_agent.invoke({"messages": [HumanMessage(content="Where is my order HT-1002?")]})

for m in result["messages"]:
    m.pretty_print()

In [ ]:
result = support_agent.invoke({"messages": [HumanMessage(content="Can I still return the headphones from order HT-1001?")]})
print(result["messages"][-1].text)

In [ ]:
result = support_agent.invoke({"messages": [HumanMessage(content="What are your support hours?")]})
print(result["messages"][-1].text)

In [ ]:
# Guardrail check: off-topic request.
result = support_agent.invoke({"messages": [HumanMessage(content="Write my history essay for me.")]})
print(result["messages"][-1].text)

In [ ]:
# Escalation check.
result = support_agent.invoke({"messages": [HumanMessage(content="This is the third time I am asking about my missing package and nobody helps. I want to talk to a person.")]})

for m in result["messages"]:
    m.pretty_print()

## 6. Memory

Try a natural follow-up and watch it fail:

In [ ]:
result = support_agent.invoke({"messages": [HumanMessage(content="Where is my order HT-1002?")]})
print(result["messages"][-1].text)
print("---")
result = support_agent.invoke({"messages": [HumanMessage(content="And when will it arrive?")]})
print(result["messages"][-1].text)

Each `invoke` starts from an empty state, so the agent has no idea what "it" refers to. LangGraph fixes this with a **checkpointer**: the graph saves its state after every step, keyed by a `thread_id`, and resumes from it on the next call.

- https://docs.langchain.com/oss/python/langgraph/persistence

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
support_agent = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "customer-1"}}

In [ ]:
result = support_agent.invoke({"messages": [HumanMessage(content="Where is my order HT-1002?")]}, config)
print(result["messages"][-1].text)
print("---")
result = support_agent.invoke({"messages": [HumanMessage(content="And when will it arrive?")]}, config)
print(result["messages"][-1].text)

In [ ]:
# A different thread_id is a different customer: no shared history.
other = {"configurable": {"thread_id": "customer-2"}}
result = support_agent.invoke({"messages": [HumanMessage(content="When will it arrive?")]}, other)
print(result["messages"][-1].text)

## 7. Chat with your agent

Uncomment and run to talk to the agent live. Type `quit` to stop.

In [ ]:
# while True:
#     user = input("You: ")
#     if user.lower() in {"quit", "exit"}:
#         break
#     result = support_agent.invoke({"messages": [HumanMessage(content=user)]}, config)
#     print("Agent:", result["messages"][-1].text)

## 8. ICA: add order cancellation

Husky Tech policy: an order can be cancelled only while its status is still `processing`.

1. Write a `cancel_order(order_id)` tool that enforces that rule and updates the order's status to `cancelled`.
2. Add it to the tool list and rebuild the graph (keep the checkpointer).
3. Test: cancelling HT-1003 should succeed; cancelling HT-1002 should be refused; cancelling HT-1003 twice should be refused the second time.
4. Does the system prompt need a new rule? Add one if so.

In [ ]:
@tool
def cancel_order(order_id: str) -> str:
    """TODO: describe what this tool does, its rule, and the argument."""
    # TODO: look the order up; handle the missing-order case
    # TODO: refuse unless status is 'processing'
    # TODO: set the status to 'cancelled' and confirm
    pass


In [ ]:
#TODO: rebuild tools, model_with_tools, and the graph with the checkpointer, then run the three tests.
